# SiPM Pulse Analysis (from `sipm_hits.root`)
This notebook builds simple, noise-free SiPM pulse estimates from optical photon hits.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import uproot

## Configuration
Adjust these parameters as needed.

In [ ]:
# Input file
root_file = Path("sipm_hits.root")

# Geometry (match module-sim.py)
pix = 3.0   # mm
foil = 0.2  # mm
n = 8
pitch = pix + foil
offset = (n - 1) * pitch / 2.0

# SiPM model (noise-free)
PDE = 0.30
rng_seed = 123

# Pulse model
# h(t) = (1 - exp(-t/τr)) * exp(-t/τd)
# (arbitrary amplitude; total charge proportional to N_pe)
tau_rise_ns = 0.5
tau_decay_ns = 12.0

# Pulse sampling
bin_width_ns = 0.1
window_ns = 200.0

## Load ROOT and build dataframe

In [ ]:
if not root_file.exists():
    roots = sorted(Path('.').glob('*.root'))
    raise FileNotFoundError(f"{root_file} not found. Available: {[p.name for p in roots]}")

with uproot.open(root_file) as f:
    classnames = f.classnames()
    tree_key = next((k for k, v in classnames.items() if v.endswith('TTree')), None)
    if tree_key is None:
        raise RuntimeError(f'No TTree found in {root_file}. Keys: {list(classnames.keys())}')
    tree = f[tree_key]
    arrays = tree.arrays(library='np')

print("Branches:" + "".join(list(arrays.keys())))

def _get(name):
    return arrays[name] if name in arrays else None

def _get_vec(base):
    arr = _get(base)
    if arr is not None:
        return arr
    x = _get(f"{base}_X")
    y = _get(f"{base}_Y")
    z = _get(f"{base}_Z")
    if x is None or y is None or z is None:
        return None
    return np.stack([x, y, z], axis=1)

pos = _get_vec('Position')
if pos is None:
    pos = _get_vec('PrePosition')
if pos is None:
    pos = _get_vec('PostPosition')
if pos is None:
    raise RuntimeError('No Position/PrePosition/PostPosition found in ROOT file')

x_mm, y_mm, z_mm = pos[:, 0], pos[:, 1], pos[:, 2]

# time (assumed ns)
t_ns = _get('GlobalTime')
if t_ns is None:
    t_ns = _get('Time')

# event id
event = _get('EventID')
if event is None:
    raise RuntimeError('EventID not found in ROOT file')

particle = _get('ParticleName')
if particle is None:
    particle = np.array(['' for _ in range(len(event))])
else:
    particle = particle.astype(str)

# Build dataframe
nrows = len(event)
df = pd.DataFrame({
    'event': event,
    't_ns': t_ns if t_ns is not None else np.full(nrows, np.nan),
    'x_mm': x_mm,
    'y_mm': y_mm,
    'z_mm': z_mm,
    'particle': particle,
})

print(df.head())

## Map hits to SiPM pixel indices (by position)

In [ ]:
# Assign to nearest pixel center by XY
ix = np.round((df['x_mm'].to_numpy() + offset) / pitch).astype(int) # the x index of the associated pitch
iy = np.round((df['y_mm'].to_numpy() + offset) / pitch).astype(int) # the y index of the associated pitch

cx = ix * pitch - offset # the x-centre of the associated pitch
cy = iy * pitch - offset # the y-centre of the associated pitch

in_bounds = (ix >= 0) & (ix < n) & (iy >= 0) & (iy < n) #check if pitch is within array
#check if positions are within pixels
inside_pixel = in_bounds & (np.abs(df['x_mm'].to_numpy() - cx) <= pix / 2.0) & (np.abs(df['y_mm'].to_numpy() - cy) <= pix / 2.0)

# Keep only hits that fall inside a SiPM pixel
mapped = df[inside_pixel].copy()
mapped['ix'] = ix[inside_pixel]
mapped['iy'] = iy[inside_pixel]

print('Total hits:', len(df))
print('Mapped to SiPM pixels:', len(mapped))

## Per-event, per-pixel photon counts and photoelectrons

In [ ]:
# Count photons per event/pixel
counts = (
    mapped.groupby(['event', 'ix', 'iy'])
    .size()
    .reset_index(name='n_photons')
)

# Convert to photoelectrons (noise-free)
rng = np.random.default_rng(rng_seed)
counts['n_pe'] = rng.binomial(counts['n_photons'].to_numpy(), PDE)

print(len(counts))

## Example: visualize photon/PE map for one event

In [ ]:
# Pick an event to inspect
if len(counts) == 0:
    raise RuntimeError('No mapped photons found for any event')

event_id = int(counts['event'].iloc[5])
event_id = 0

# Build 2D maps
map_ph = np.zeros((n, n), dtype=float)
map_pe = np.zeros((n, n), dtype=float)

sub = counts[counts['event'] == event_id]
for _, row in sub.iterrows():
    map_ph[int(row['iy']), int(row['ix'])] = row['n_photons']
    map_pe[int(row['iy']), int(row['ix'])] = row['n_pe']

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(map_ph, origin='lower', cmap='magma')
ax[0].set_title(f'Photon map (event {event_id})')
ax[0].set_xlabel('ix')
ax[0].set_ylabel('iy')
plt.colorbar(ax[0].images[0], ax=ax[0])

ax[1].imshow(map_pe, origin='lower', cmap='viridis')
ax[1].set_title(f'PE map (event {event_id})')
ax[1].set_xlabel('ix')
ax[1].set_ylabel('iy')
plt.colorbar(ax[1].images[0], ax=ax[1])
plt.tight_layout()
plt.show()

## Pulse model for one pixel

In [ ]:
# Select a pixel within the event to build a pulse
ix_sel, iy_sel = int(sub['ix'].iloc[0]), int(sub['iy'].iloc[0])

hits_sel = mapped[(mapped['event'] == event_id) & (mapped['ix'] == ix_sel) & (mapped['iy'] == iy_sel)]

if len(hits_sel) == 0:
    raise RuntimeError('No hits for selected event/pixel')

# Convert photons to PE by randomly selecting PDE fraction
pe_mask = rng.random(len(hits_sel)) < PDE
hit_times = hits_sel['t_ns'].to_numpy()
pe_times = hit_times[pe_mask]

# Build histogram in time bins
if len(pe_times) == 0:
    print('No PE for this pixel after PDE; increase PDE or pick another event/pixel')
else:
    t0 = pe_times.min()
    tmax = t0 + window_ns
    bins = np.arange(t0, tmax + bin_width_ns, bin_width_ns)
    hist, edges = np.histogram(pe_times, bins=bins)

    # Single-PE response kernel
    t_kernel = np.arange(0, window_ns, bin_width_ns)
    h = (1 - np.exp(-t_kernel / tau_rise_ns)) * np.exp(-t_kernel / tau_decay_ns)

    # Convolve to build pulse
    pulse = np.convolve(hist, h, mode='full')[:len(hist)]

    t_axis = edges[:-1] - t0

    plt.figure(figsize=(6, 4))
    plt.plot(t_axis, pulse, label='Pulse')
    plt.xlabel('Time (ns)')
    plt.ylabel('Arbitrary units')
    plt.title(f'Pulse: event {event_id}, pixel ({ix_sel},{iy_sel})')
    plt.tight_layout()
    plt.show()

## Simple DOI-sensitive features (per event)

In [ ]:
# Features: total PE, max-pixel fraction, spatial spread
# Build per-event total PE map
features = []
for evt, grp in counts.groupby('event'):
    total_pe = grp['n_pe'].sum()
    if total_pe == 0:
        continue
    max_pe = grp['n_pe'].max()
    frac_max = max_pe / total_pe

    # light spread (RMS of pixel positions weighted by PE)
    xs = grp['ix'].to_numpy()
    ys = grp['iy'].to_numpy()
    w = grp['n_pe'].to_numpy()
    cx = np.average(xs, weights=w)
    cy = np.average(ys, weights=w)
    spread = np.sqrt(np.average((xs - cx)**2 + (ys - cy)**2, weights=w))

    features.append({'event': evt, 'total_pe': total_pe, 'max_pe': max_pe, 'frac_max': frac_max, 'spread': spread})

feat = pd.DataFrame(features)

# Add per-pixel PE columns (wide format)
pe_wide = counts.pivot_table(index='event', columns=['ix','iy'], values='n_pe', aggfunc='sum', fill_value=0)
pe_wide.columns = [f'pe_{ix}_{iy}' for ix, iy in pe_wide.columns]
pe_wide = pe_wide.reset_index()
feat = feat.merge(pe_wide, on='event', how='left')
print(feat.head())

# Optional PE window for plots (min, max). Use None to disable.
pe_window = (None,None)  # e.g., (200, 5000)
plot_feat = feat
if pe_window is not None:
    lo, hi = pe_window
    if lo is not None:
        plot_feat = plot_feat[plot_feat['max_pe'] >= lo]
    if hi is not None:
        plot_feat = plot_feat[plot_feat['max_pe'] <= hi]

# Histogram: total PE per event
if len(plot_feat) == 0:
    print('No events with PE found')
else:
    plt.figure(figsize=(6, 4))
    plt.hist(plot_feat['total_pe'], bins=40, alpha=0.8)
    plt.xlabel('Total photoelectrons per event')
    plt.ylabel('Events')
    plt.title('Total PE across events')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.hist(plot_feat['max_pe'], bins=30, alpha=0.8)
    plt.xlabel('Max pixel ')
    plt.ylabel('Events')
    plt.title('Max pixel')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.hist(plot_feat['frac_max'], bins=30, alpha=0.8)
    plt.xlabel('Max pixel fraction')
    plt.ylabel('Events')
    plt.title('Max pixel fraction (DOI-sensitive)')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.hist(plot_feat['spread'], bins=30, alpha=0.8)
    plt.xlabel('Spatial spread (RMS)')
    plt.ylabel('Events')
    plt.title('Light spread (DOI-sensitive)')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.scatter(plot_feat['frac_max'], plot_feat['spread'], s=8, alpha=0.5)
    plt.xlabel('Max pixel fraction')
    plt.ylabel('Spatial spread (RMS)')
    plt.title('Max pixel fraction vs light spread')
    plt.tight_layout()
    plt.show()

In [ ]:
# Join gamma_steps.root to add primary interaction position (DOI truth)
from pathlib import Path

gamma_file = Path('gamma_steps.root')
if not gamma_file.exists():
    print('gamma_steps.root not found; run module-sim.py with gamma_steps actor first.')
else:
    with uproot.open(gamma_file) as f:
        classnames = f.classnames()
        tree_key = next((k for k, v in classnames.items() if v.endswith('TTree')), None)
        if tree_key is None:
            raise RuntimeError(f'No TTree found in {gamma_file}. Keys: {list(classnames.keys())}')
        tree = f[tree_key]
        g = tree.arrays(library='np')

    def _gget(name):
        return g[name] if name in g else None

    def _gget_vec(base):
        arr = _gget(base)
        if arr is not None:
            return arr
        x = _gget(f"{base}_X")
        y = _gget(f"{base}_Y")
        z = _gget(f"{base}_Z")
        if x is None or y is None or z is None:
            return None
        return np.stack([x, y, z], axis=1)

    g_event = _gget('EventID')
    g_parent = _gget('ParentID')
    g_particle = _gget('ParticleName')
    g_process = _gget('ProcessDefinedStep')
    g_pos = _gget_vec('PostPosition')

    if g_event is None or g_pos is None:
        raise RuntimeError('gamma_steps.root missing EventID or Position/PrePosition/PostPosition')

    # Build gamma steps dataframe
    gdf = pd.DataFrame({
        'event': g_event,
        'parent': g_parent if g_parent is not None else np.nan,
        'particle': g_particle.astype(str) if g_particle is not None else '',
        'process': g_process.astype(str) if g_process is not None else '',
        'x_gamma': g_pos[:, 0],
        'y_gamma': g_pos[:, 1],
        'z_gamma': g_pos[:, 2]
    })

    # Primary gamma first interaction: ParentID==0 and process != Transportation
    mask_primary = (gdf['parent'] == 0) & (~gdf['process'].str.contains('transport', case=False, na=False))
    gdf_primary = gdf.copy()

    #print(gdf_primary)

    # First interaction per event
    gdf_primary = gdf_primary.groupby('event').first().reset_index()

    # Join to feature table (feat) if available
    if 'feat' in globals():
        feat = feat.merge(gdf_primary[['event','x_gamma','y_gamma','z_gamma']], on='event', how='left')

        print('Joined gamma interaction positions to feat')
        display(feat.head())
    else:
        print('feat not found; run the features cell first.')


In [ ]:
# Scatter: z_gamma vs DOI variables

# Optional x-axis limits for z_gamma (set to None to auto)
xlim_z = (-10, 10.0)  # e.g., (-10, 10)

# Optional PE window (min, max). Use None to disable.
pe_window = (550, 1400)  # e.g., (200, 5000)
frac_max_window = (0.49, 1.0)  # this should reject scatter events

if 'feat' not in globals() or not set(['z_gamma','frac_max','spread','total_pe']).issubset(feat.columns):
    print('feat with z_gamma/frac_max/spread/total_pe not found; run join cell first.')
else:
    plot_df = feat.copy()
    if pe_window is not None:
        lo, hi = pe_window
        if lo is not None:
            plot_df = plot_df[plot_df['max_pe'] >= lo]
        if hi is not None:
            plot_df = plot_df[plot_df['max_pe'] <= hi]
    if frac_max_window is not None:
        lo, hi = frac_max_window
        if lo is not None:
            plot_df = plot_df[plot_df['frac_max'] >= lo]
        if hi is not None:
            plot_df = plot_df[plot_df['frac_max'] <= hi]

    if len(plot_df) == 0:
        print('No events left after PE window.')
    else:
        # Z distribution
        plt.figure(figsize=(6, 4))
        plt.hist(plot_df['z_gamma'], bins=40, alpha=0.8)
        plt.xlabel('Z')
        plt.ylabel('N events')
        plt.tight_layout()
        plt.show()

        plots = [
            ('frac_max', 'Max Pixel Fraction'),
            ('spread', 'Light Spread (RMS)'),
            ('total_pe', 'Total PE'),
            ('max_pe', 'Max PE'),
        ]

        for col, label in plots:
            plt.figure(figsize=(6, 4))
            plt.scatter(plot_df['z_gamma'], plot_df[col], s=8, alpha=0.4)
            plt.xlabel('Primary Interaction Z (mm)')
            if xlim_z is not None:
                plt.xlim(xlim_z)

            plt.ylabel(label)
            plt.title(f'{label} vs Z (DOI)')
            plt.tight_layout()
            plt.show()

        # Ratio plot (if you want to keep it)
        plt.figure(figsize=(6, 4))
        plt.scatter(plot_df['z_gamma'], (plot_df['frac_max'] / plot_df['spread']), s=8, alpha=0.4)
        plt.xlabel('Primary Interaction Z (mm)')
        plt.ylabel('frac_max / spread')
        plt.ylim(0.2, 1.0)
        if xlim_z is not None:
            plt.xlim(xlim_z)
        plt.tight_layout()
        plt.show()

        # 1D distributions of max pixel fraction for Z bins
        if xlim_z is None:
            zmin, zmax = plot_df['z_gamma'].min(), plot_df['z_gamma'].max()
        else:
            zmin, zmax = xlim_z
        if zmin == zmax:
            print('Z range is zero; skipping binned distributions.')
        else:
            bins = np.linspace(zmin, zmax, 7)  # 6 equal ranges
            plt.figure(figsize=(7, 4))
            for i in range(6):
                lo, hi = bins[i], bins[i+1]
                mask = (plot_df['z_gamma'] >= lo) & (plot_df['z_gamma'] < hi)
                if mask.sum() == 0:
                    continue
                plt.hist(
                    plot_df.loc[mask, 'frac_max'],
                    bins = np.linspace(0.5, 0.7, 30),
                    density=True,
                    histtype='bar', alpha=0.35, edgecolor = 'k',
                    label=f'{lo:.2f} to {hi:.2f} mm',
                )
            plt.xlabel('Max pixel fraction')
            plt.ylabel('Density')
            plt.title('Max pixel fraction by Z bin')
            plt.legend(fontsize=8)
            plt.tight_layout()
            plt.show()


        # Mean max pixel fraction vs Z bin centers (with 1-sigma error bars)
        if xlim_z is None:
            zmin, zmax = plot_df['z_gamma'].min(), plot_df['z_gamma'].max()
        else:
            zmin, zmax = xlim_z
        if zmin == zmax:
            print('Z range is zero; skipping mean-by-bin plot.')
        else:
            bins = np.linspace(zmin, zmax, 7)  # 6 equal ranges
            centers = 0.5 * (bins[:-1] + bins[1:])
            means = []
            sigmas = []
            ns = []
            sems = []
            for i in range(6):
                lo, hi = bins[i], bins[i+1]
                vals = plot_df.loc[(plot_df['z_gamma'] >= lo) & (plot_df['z_gamma'] < hi), 'frac_max']
                if len(vals) == 0:
                    means.append(np.nan)
                    sigmas.append(np.nan)
                    ns.append(0)
                    sems.append(np.nan)
                else:
                    means.append(vals.mean())
                    sigmas.append(vals.std())
                    n = len(vals)
                    ns.append(n)
                    sems.append(vals.std() / np.sqrt(n) if n > 1 else np.nan)

            plt.figure(figsize=(6, 4))
            plt.errorbar(centers, means, yerr=sems, fmt='o-', capsize=3)
            plt.xlabel('Z bin center (mm)')
            plt.ylabel('Mean max pixel fraction')
            plt.title('Mean max pixel fraction vs Z bin')
            plt.tight_layout()
            plt.show()

        # Linear fit to mean max pixel value vs Z bin center
        # Use only finite points
        centers_arr = np.array(centers)
        means_arr = np.array(means)
        sems_arr = np.array(sems)
        mask_fit = np.isfinite(centers_arr) & np.isfinite(means_arr) & np.isfinite(sems_arr) & (sems_arr > 0)
        if mask_fit.sum() < 2:
            print('Not enough points for linear fit.')
        else:
            w = 1.0 / (sems_arr[mask_fit] ** 2)
            x = centers_arr[mask_fit]
            y = means_arr[mask_fit]
            S = np.sum(w)
            Sx = np.sum(w * x)
            Sy = np.sum(w * y)
            Sxx = np.sum(w * x * x)
            Sxy = np.sum(w * x * y)
            den = S * Sxx - Sx * Sx
            if den == 0:
                print('Degenerate fit (den=0); cannot fit.')
                m, b = np.nan, np.nan
            else:
                m = (S * Sxy - Sx * Sy) / den
                b = (Sy - m * Sx) / S
            plt.figure(figsize=(6, 4))
            plt.errorbar(centers_arr, means_arr, yerr=sems_arr, fmt='o', capsize=3, label='Means')
            xfit = np.linspace(centers_arr[mask_fit].min(), centers_arr[mask_fit].max(), 100)
            yfit = m * xfit + b
            plt.plot(xfit, yfit, '-', label=f'Fit: y = {m:.3g} x + {b:.3g}')
            plt.xlabel('Z bin center (mm)')
            plt.ylabel('Mean max pixel (PE)')
            plt.title('Linear fit: mean max pixel vs Z')
            plt.legend()
            plt.tight_layout()
            plt.show()

            # Estimate Z from max_pe using inverse of fitted line
            if m == 0:
                print('Fit slope is zero; cannot invert for Z.')
            else:
                z_est = (plot_df['frac_max'] - b) / m
                dz = plot_df['z_gamma'] - z_est
                plt.figure(figsize=(6, 4))
                counts, edges, _ = plt.hist(dz, bins=50, alpha=0.85)
                plt.xlabel('Z_true - Z_est (mm)')
                plt.ylabel('Counts')
                plt.title('Z residuals from linear frac_max->Z inversion')

                # Gaussian fit to residuals (counts vs bin centers)
                centers = 0.5 * (edges[:-1] + edges[1:])
                def _gauss(x, A, mu, sigma):
                    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

                A0 = counts.max() if len(counts) > 0 else 1.0
                mu0 = float(np.mean(dz)) if len(dz) > 0 else 0.0
                sigma0 = float(np.std(dz)) if len(dz) > 0 else 1.0

                mu_err = np.nan
                fit_ok = False
                try:
                    from scipy.optimize import curve_fit
                    popt, pcov = curve_fit(_gauss, centers, counts, p0=[A0, mu0, sigma0], maxfev=20000)
                    A, mu, sigma = popt
                    perr = np.sqrt(np.diag(pcov))
                    mu_err = perr[1] if len(perr) > 1 else np.nan
                    fit_ok = True
                except Exception:
                    # Fallback: moment estimates if scipy not available or fit fails
                    A, mu, sigma = A0, mu0, sigma0
                    if len(dz) > 0 and sigma0 > 0:
                        mu_err = sigma0 / np.sqrt(len(dz))

                fwhm = 2.355 * sigma if sigma is not None else np.nan

                xfit = np.linspace(edges[0], edges[-1], 200) if len(edges) > 1 else np.array([0.0, 1.0])
                yfit = _gauss(xfit, A, mu, sigma) if sigma is not None and sigma != 0 else np.zeros_like(xfit)
                plt.plot(xfit, yfit, 'r-', label='Gaussian fit' if fit_ok else 'Gaussian (moment est.)')

                txt = f"$\mu$ = {mu:.3g} ± {mu_err:.2g}\n$\sigma$ = {sigma:.3g}\nFWHM = {fwhm:.3g}"
                plt.text(0.98, 0.95, txt, transform=plt.gca().transAxes, ha='right', va='top',
                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)
                plt.legend(fontsize=8, loc='best')

                plt.tight_layout()
                plt.show()
